# Warm start experiment (CARLA)

Each new flight means a retrain. This notebook asks **where that retrain should start from**:

| setup | round *k* starts from |
|---|---|
| `cold` | the COCO-pretrained weights, every round |
| `warm` | round *k-1*'s `last.pt` |
| `sp` | Shrink & Perturb on round *k-1*'s `last.pt` (θ ← λθ + 0.01·θ_random, λ = 0.6) |

Rounds are flights, arriving in order and keeping the archive:

| round | flights | tiles |
|---|---|---|
| 1 | 17 Nov 2025 | 704 |
| 2 | + 5 Dec 2025 | 1600 |
| 3 | + 11 Dec 2025 | 1864 |
| **4** | **+ 15 May 2025 high flight** | **2688** |

Rounds 1–3 are all low flights over the same field, so there is barely any
distribution shift and the three setups came out within 1 mAP50 point of each
other. **Round 4 is the real test:** a model trained on the low flights scores
mAP50 **0.011** on the high flight, so it is a condition it has never seen, and
that is where warm starting, cold restarts and Shrink & Perturb should actually
diverge.

Each round is then scored on every test set **separately** (`evaluate_matrix.py`),
because one pooled number cannot separate the two things that matter:

- **plasticity** — did the model learn the flight that just arrived? (`high_1080p` at round 4)
- **retention** — did it stay good on the flights it already knew? (`low_*` at round 4)

`cold` is the reference, not a competitor: retraining from scratch on everything
is what you would do with unlimited compute. Shrink & Perturb earns its place
only if it matches `cold` **at a lower epoch budget**, which is what the
`ep10` variant below measures.

---

**Before starting:** `Runtime → Change runtime type → T4 GPU`.

**On Drive, in `MyDrive/TFG/`:**
`carla-v1-clean.zip`, `high-altitude-tiles-1080p.zip`, `high-altitude-tiles-4k.zip` (optional),
`make_rounds.py`, `shrink_perturb.py`, `run_experiment.py`, `evaluate_matrix.py`, `summarize.py`.

**After a disconnect:** run all cells again from the top. Finished jobs are skipped,
so the run continues where it stopped.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")
TFG = "/content/drive/MyDrive/TFG"
RESULTS = f"{TFG}/results"                        # the main experiment
RESULTS_HIGH_FIRST = f"{TFG}/results_high_first"  # the reverse stream (optional)
# everything that must survive a disconnect goes under TFG

In [ ]:
# 3. Same Ultralytics version as the laptop and the original thesis
!pip install -q ultralytics==8.4.14
import torch, ultralytics
print("torch", torch.__version__, "| ultralytics", ultralytics.__version__)

In [ ]:
# 4. Copy the datasets to the local disk (training from Drive is slow) and build the rounds.
#    /content is wiped when the session ends, so this runs every session. The round files
#    hold absolute paths, which is why they have to be rebuilt here and not on the laptop.
import os

LOW = "/content/carla-v1"
HIGH = "/content/high-altitude-tiles-1080p"
HIGH_4K = "/content/high-altitude-tiles-4k"   # only used by one optional test set

for name, path in [("carla-v1-clean", LOW),
                   ("high-altitude-tiles-1080p", HIGH),
                   ("high-altitude-tiles-4k", HIGH_4K)]:
    if not os.path.exists(f"{path}/test/images") and os.path.exists(f"{TFG}/{name}.zip"):
        !cp {TFG}/{name}.zip /content/
        !unzip -q /content/{name}.zip -d /content/

!python {TFG}/make_rounds.py --low {LOW} --high {HIGH} --out /content/rounds
# Expected: 704 / 1600 / 1864 / 2688 tiles

## Optional: quick test (~6 min)

Runs one job per setup for 1 epoch on 10% of the tiles, into a throwaway folder.
Use it once per session change to check nothing is broken before the real run.

In [ ]:
!python {TFG}/run_experiment.py --rounds-dir /content/rounds --out /content/quick_test \
    --seeds 42 --epochs 1 --fraction 0.1

## The real run

30 jobs in total (3 seeds × [shared round 1 + 3 setups × rounds 2–4]), of which
**17 are already finished** on Drive from the earlier rounds, so this cell has
13 to do: the 4 leftover seed-44 jobs plus round 4 for all three setups.
About **6 hours** on a T4, so it will take more than one session.

Keep the browser tab open while it runs; Colab disconnects idle sessions.

In [ ]:
!python {TFG}/run_experiment.py --rounds-dir /content/rounds --out {RESULTS} \
    --seeds 42 43 44 --epochs 30 --lam 0.6 --noise-scale 0.01

## Variants (run after the main run)

Each one writes to its own folder via `--tag`, so nothing above is overwritten,
and each continues from the main experiment's round 3.

1. **Epoch budget** — `warm` and `sp` at 10 epochs against `cold` at 30. This is
   the only comparison in which Shrink & Perturb can be *useful* rather than
   merely harmless: if it needs the same 30 epochs as a cold restart, the honest
   conclusion is to always retrain from scratch. 6 jobs, ~1 h.
2. **How much to shrink** — λ is the retention/plasticity dial (λ→1 is warm,
   λ→0 is cold), and it only has something to trade off at a shift round. 6 jobs, ~3 h.
3. **High flight alone** — round 1 of the reverse stream, trained on the 824 high
   tiles from COCO. Needed to read round 4's high score at all: without it there
   is no way to tell whether 0.011 → 0.55 is good or bad. 3 jobs, ~0.5 h.

In [ ]:
# 1. Epoch budget: warm and sp get 10 epochs, cold above had 30
!python {TFG}/run_experiment.py --rounds-dir /content/rounds --out {RESULTS} \
    --rounds 4 --setups warm sp --epochs 10 --tag ep10 --seeds 42 43 44

In [ ]:
# 2. Shrink factor, at the shift round only (the chain up to round 3 stays lambda = 0.6)
!python {TFG}/run_experiment.py --rounds-dir /content/rounds --out {RESULTS} \
    --rounds 4 --setups sp --lam 0.3 --tag lam0.3 --seeds 42 43 44
!python {TFG}/run_experiment.py --rounds-dir /content/rounds --out {RESULTS} \
    --rounds 4 --setups sp --lam 0.9 --tag lam0.9 --seeds 42 43 44

In [ ]:
# 3. High flight on its own = round 1 of the stream that starts with the high flight.
#    Running more rounds of this folder later gives a second, independent shift event
#    (low flights arriving after a high one) out of the same data.
!python {TFG}/make_rounds.py --low {LOW} --high {HIGH} --order high-first --out /content/rounds_high_first
!python {TFG}/run_experiment.py --rounds-dir /content/rounds_high_first --out {RESULTS_HIGH_FIRST} \
    --rounds 1 --epochs 30 --seeds 42 43 44

## The evaluation matrix

Scores every finished job on each test set on its own. It only loads checkpoints,
so the 17 jobs trained earlier get the new measurement without being retrained,
and it is safe to stop and rerun: sets already done are skipped.

| test set | tiles | |
|---|---|---|
| `low_all` | 696 | the three low flights pooled (the number reported so far) |
| `low_20251117` / `low_20251205` / `low_20251211` | 264 / 320 / 112 | the same tiles per flight day |
| `high_1080p` | 264 | the high flight through the same pipeline as its training tiles |
| `high_4k` | 1320 | the high flight tiled at native 4K, so plants are ~1.7× bigger than in training |

~30 min for all checkpoints on a T4.

In [ ]:
!python {TFG}/evaluate_matrix.py --results {RESULTS} \
    --low {LOW} --high-1080p {HIGH} --high-4k {HIGH_4K} \
    --eval-sets /content/eval_sets --device 0

## Results

Can be run at any time, also with only some jobs finished.

- `summary.csv` — the pooled low-flight number per (setup, round), as before
- `matrix.csv` — (setup, round) × test set, mean ± std over seeds
- `paired.csv` — per-seed differences between setups

The paired table is the one to trust: the seed-to-seed spread of a single setup
(~0.004 mAP50) is about the size of the gap between setups, so subtracting two
setups *within the same seed* removes the noise they share. Three seeds pointing
the same way is weak but real evidence; two up and one down is not.

In [ ]:
!python {TFG}/summarize.py --results {RESULTS}

In [ ]:
# The variants against the cold reference
!python {TFG}/summarize.py --results {RESULTS} \
    --pairs sp:cold sp:warm sp_ep10:cold warm_ep10:cold sp_lam0.3:sp sp_lam0.9:sp

# The high-flight-only baseline
!python {TFG}/summarize.py --results {RESULTS_HIGH_FIRST}

## Export the results to the repo

Collects only the text files that describe each job (`results.json`,
`eval_matrix.json`, `results.csv`, `args.yaml`) into one small zip on Drive.
The checkpoints stay here: ~5 MB each, rewritten on every run, and nothing in
the analysis reads them once `eval_matrix.json` exists.

Download `TFG/results_export.zip` from Drive, then on the laptop:

```bash
python experiments/warm_start/export_results.py \
    --unzip ~/Downloads/results_export.zip --dest experiments/warm_start/results
```


In [ ]:
!python {TFG}/export_results.py --src {RESULTS} {RESULTS_HIGH_FIRST} \
    --zip {TFG}/results_export.zip